## LPD-26158

**Takeaways:**
* Raw backup data stored in GCS (bak_data) includes Bot/Crawlers and events that Dataflow is not able to parse
* Batch Dataflow returns an inconsistent number of events across different runs on the same data. Exceptions happening during the batch event processing
* Batch Dataflow parsed data (json_data) is generally less than Streaming Dataflow parsed data.
* Session id generation matches only partially.


In [ ]:
google_project = 'liferaycloud-ac3'
google_region = 'europe-west3'

external_postgres_connection = f'{google_project}.{google_region}.postgresql'

In [ ]:
import os

if os.environ.get('PYTHONPATH') is None:
    os.environ['PYTHONPATH']=''

In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

def get_spark_session(application_name=None, spark_conf=None):
    conf_args = '--conf spark.driver.extraJavaOptions="-Dio.netty.tryReflectionSetAccessible=true -XX:ThreadStackSize=8192"'
    # https://github.com/apache/arrow/pull/4522

    jar_jargs = '--jars=gs://spark-lib/bigquery/spark-bigquery-latest_2.12.jar'

    command = 'pyspark-shell'

    os.environ['PYSPARK_SUBMIT_ARGS'] = " ".join([conf_args, jar_jargs, command])

    spark_session_builder = SparkSession.Builder()

    spark_session_builder = spark_session_builder.config(conf=spark_conf or get_spark_conf(application_name))

    spark = spark_session_builder.getOrCreate()
    
    return spark

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

dataset_id = "{}.lpd_26158".format(client.project)

dataset = client.get_dataset(dataset_id)

if dataset is None:
    # Construct a full Dataset object to send to the API.
    dataset = bigquery.Dataset(dataset_id)

    dataset.location = google_region

    # Send the dataset to the API for creation, with an explicit timeout.
    # Raises google.api_core.exceptions.Conflict if the Dataset already
    # exists within the project.

    dataset = client.create_dataset(dataset, timeout=30)  # Make an API request.

    print("Created dataset {}.{}".format(client.project, dataset.dataset_id))

materializationDataset = dataset.dataset_id

In [ ]:
import os
import sys

sys.path.append(os.environ['HOME'] + '/libs/')

# from notebook_common import *

app_name = "LRAC-26158"

spark_conf=SparkConf() #get_spark_conf(application_name=app_name, master='yarn')

spark_conf.setAppName(app_name)
spark_conf.setMaster('yarn')

spark_conf.set('spark.jars', 'gs://spark-lib/bigquery/spark-bigquery-latest_2.12.jar')
spark_conf.set('spark.sql.shuffle.partitions', 128)

spark_conf.set('materializationDataset', materializationDataset)
spark_conf.set('viewsEnabled', 'true')
spark_conf.set('temporaryGcsBucket', 'ac-interest-score/temp_write_bucket_for_bq_writes')

spark_conf.set('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
spark_conf.set('spark.kryoserializer.buffer.max', '2000m')

spark_conf.set('spark.driver.maxResultSize', '0')

spark_conf.set('spark.executor.cores', '2')
spark_conf.set('spark.executor.memory', '9g')

spark = get_spark_session(application_name=app_name, spark_conf=spark_conf)
spark

In [ ]:
import pyspark.sql.functions as F

In [ ]:
event_date = '2024-05-16'

---
## Inspecting RAW Analytics Events from GCS backup

Analytics Cloud `eventingestionpipeline` has a `Backup PubSub Messages` stage. It writes raw messages incoming from PubSub and stores them into a designated GCS bucket. The purpose of this inspection is to check how many events were received via PubSub on a given date. This helps validate following steps of the data validation by comparing numbers of the effective events at each point in time of the pipelines.

Apache Spark uses Hadoop libraries to read from GCS and is affected by a bug that does not allow to have `:` in file names. We are going to implement a workaournd as described [here](https://stackoverflow.com/a/68498839)  to read the raw Events data.

In [ ]:
path = f'gs://{google_project}-analytics-events/{event_date}/*'  # can include filenames with with ':'
paths_file = "events_bak_path_list.txt"

!gsutil ls -r $path > $paths_file

path_list = open(paths_file, 'r').read().split("\n")[:-1]

json_events_bak = spark.read.json(path_list)
json_events_bak.cache()

json_events_bak_count = json_events_bak.count()
json_events_bak_count

There are 1.9M of raw events, however this includes events generated by crawlers

In [ ]:
json_events_bak.distinct().count()

In [ ]:
json_events_bak.select('id').distinct().count()

Also we note there are few duplicated events. This is normal due to PubSub.

We download the [browscap](https://browscap.org/) database as used in our Dataflow pipeline. Unfortunately it seems the official python library is outdated however the parsing should be as easy as describede [here](https://www.djangosnippets.org/snippets/267/)

In [ ]:
!curl http://browscap.org/stream\?q\=BrowsCapINI > /home/riccardoferrari/browscap.ini

In [ ]:
from configparser import ConfigParser
import re

class Browser(object):

    def __init__(self, capabilities):
        self.lazy_flag = True
        self.cap = capabilities


    def parse(self):
        for name, value in self.cap.items():
            if name in ["tables", "aol", "javaapplets",
                       "activexcontrols", "backgroundsounds",
                       "vbscript", "win16", "javascript", "cdf",
                       "wap", "crawler", "netclr", "beta",
                        "iframes", "frames", "stripper", "wap"]:
                self.cap[name] = (value.strip().lower() == "true")
            elif name in ["ecmascriptversion", "w3cdomversion"]:
                self.cap[name] = float(value)
            elif name in ["css"]:
                self.cap[name] = int(value)
            else:
                self.cap[name] = value
        self.lazy_flag = False


    def __repr__(self):
        if self.lazy_flag: self.parse()
        return repr(self.cap)


    def get(self, name, default=None):
        if self.lazy_flag: self.parse()
        try:
            return self[name]
        except KeyError:
            return default


    def __getitem__(self, name):
        if self.lazy_flag: self.parse()
        return self.cap[name.lower()]


    def keys(self):
        return self.cap.keys()


    def items(self):
        if self.lazy_flag: self.parse()
        return self.cap.items()


    def values(self):
        if self.lazy_flag: self.parse()
        return self.cap.values()
    

    def __len__(self):
        return len(self.cap)


    def supports(self, feature):
        value = self.cap.get(feature)
        if value == None:
            return False
        return value


    def features(self):
        l = []
        for f in ["tables", "frames", "iframes", "javascript",
                  "cookies", "w3cdomversion", "wap"]:
            if self.supports(f):
                l.append(f)
        if self.supports_java():
            l.append("java")
        if self.supports_activex():
            l.append("activex")
        css = self.css_version()
        if css > 0:
            l.append("css1")
        if css > 1:
            l.append("css2")
        return l


    def supports_tables(self):
        return self.supports("frames")

    def supports_iframes(self):
        return self.supports("iframes")


    def supports_frames(self):
        return self.supports("frames")


    def supports_java(self):
        return self.supports("javaapplets")


    def supports_javascript(self):
        return self.supports("javascript")


    def supports_vbscript(self):
        return self.supports("vbscript")


    def supports_activex(self):
        return self.supports("activexcontrols")


    def supports_cookies(self):
        return self.supports("cookies")


    def supports_wap(self):
        return self.supports("wap")


    def css_version(self):
        return self.get("css", 0)


    def version(self):
        major = self.get("majorver")
        minor = self.get("minorver")
        if major and minor:
            return (major, minor)
        elif major:
            return (major, None)
        elif minor:
            return (None, minor)
        else:
            ver = self.get("version")
            if ver and "." in ver:
                return tuple(ver.split(".", 1))
            elif ver:
                return (ver, None)
            else:
                return (None, None)


    def dom_version(self):
        return self.get("w3cdomversion", 0)


    def is_bot(self):
        return self.get("crawler") == True


    def is_mobile(self):
        return self.get("ismobiledevice") == True

    
    def name(self):
        return self.get("browser")




class BrowserCapabilities(object):

    BC_PATH = '/home/riccardoferrari'
    
    def __new__(cls, *args, **kwargs):
        # Only create one instance of this clas
        if "instance" not in cls.__dict__:
            cls.instance = object.__new__(cls, *args, **kwargs)
        return cls.instance


    def __init__(self):
        self.cache = {}
        self.parse()


    def parse(self):
        cfg = ConfigParser()
        files = ("browscap.ini", "bupdate.ini")
        read_ok = cfg.read([os.path.join(self.BC_PATH, name) for name in files])
        if len(read_ok) == 0:
            raise Exception("Could not read browscap.ini")
            
        self.sections = []
        self.items = {}
        self.browsers = {}
        parents = set()
        for name in cfg.sections():
            qname = name
            for unsafe in list("^$()[].-"):
                qname = qname.replace(unsafe, "\%s" % unsafe)
            qname = qname.replace("?", ".").replace("*", ".*?")
            qname = "^%s$" % qname
            sec_re = re.compile(qname)
            sec = dict(regex=qname)
            sec.update(cfg.items(name))
            p = sec.get("parent")
            if p: parents.add(p)
            self.browsers[name] = sec
            if name not in parents:
                self.sections.append(sec_re)
            self.items[sec_re] = sec


    def query(self, useragent):
        b = self.cache.get(useragent)
        if b: return b

        for sec_pat in self.sections:
            if sec_pat.match(useragent):
                browser = dict(agent=useragent)
                browser.update(self.items[sec_pat])
                parent = browser.get("parent")
                while parent:
                    items = self.browsers[parent]
                    for key, value in items.items():
                        if key not in browser.keys():
                            browser[key] = value
                        elif key == "browser" and value != "DefaultProperties":
                            browser["category"] = value # Wget, Godzilla -> Download Managers
                    parent = items.get("parent")
                if browser.get("browser") != "Default Browser":
                    b = Browser(browser)
                    self.cache[useragent] = b 
                    return b
        self.cache[useragent] = None


    __call__ = query

Given the amount of events we need to optimize the crawler scanning by filtering out the events as processed by the Dataflow pipeline.

We are going to load the events as parsed by the offline `Event` processing pipeline. This pipeline already filters out crawlers and bots and we are sure we can remove all those events from the backup event list.
This will lead to a much smaller list of user agents to scan and possibly a reasonable time in finding how many crawlers are present in all the events received on the selected date.

---

## Loading data processed by the Dataflow pipeline against the backup data

We are loading the json events as generated by the Dataflow pipeline run against the backup data.

The pipeline is an exact copy of the streaming Dataflow pipeline modified to read events from the GCS and store them back to a GCS location instead of BigQuery.
The reason to store them on GCS instead of storing the into BigQuery already is to allow this analysis and make sure the new data is properly validated.

In [ ]:
path = f"gs://{google_project}-dataflow/batch/output/{event_date.replace('-', '')}/events/*.jsonl"  # can include filenames with with ':'
paths_file = "events_json_path_list.txt"

!gsutil ls -r $path > $paths_file

path_list = open(paths_file, 'r').read().split("\n")[:-1]

json_events = spark.read.json(path_list)
json_events.cache()

json_event_count = json_events.count()
json_event_count

In [ ]:
print(f"Event count difference between raw backup data and dataflow output: {json_events_bak_count - json_event_count}")

We now have to verify the number of crawlers/bot in this much smaller list of 150k events.

First we filter all the events that do not bleong to the `json_events` dataframe 

---

We are going to inspect non parsed events

In [ ]:
json_events_not_parsed = json_events_bak.join(
    json_events.selectExpr('id as id2'),
    how='left',
    on=F.col('id')==F.col('id2')
).where(
    'id2 is NULL'
).drop('id2')

json_events_not_parsed_count = json_events_not_parsed.count()
json_events_not_parsed_count

---

The following code runs on the Driver node.

We can now collect all the user agents and scan them using the `browsercap` parser

In [ ]:
uas = json_events_not_parsed.select('context.userAgent').rdd.map(lambda r: r[0]).collect()

In [ ]:
browsercap = BrowserCapabilities()

In [ ]:
from multiprocessing import Pool, Value

bot_count = Value('i', 0)

def init_pool(ct):
    global counter
    counter = ct

def is_bot(ua):
    global counter
    
    browser = browsercap.query(ua)
    
    if browser is not None and browser.is_bot():
        with counter.get_lock():
            counter.value += 1
    

with Pool(initializer=init_pool, initargs=(bot_count,), ) as pool:
    pool.map(is_bot, uas, chunksize=1000)

In [ ]:
print(f"Total BOT/Crawler count: {bot_count.value}")

In [ ]:
print(f"Total BOT/Crawler count: {bot_count.value}")

---

The event count difference between `json_events_bak` and `json_events` is stored in the `json_events_not_parsed` dataframe. This difference includes the number of events that are generated by bots.

In [ ]:
print(f"Number of events effectively not parsed: {json_events_not_parsed_count - bot_count.value}")

Those events are effectively lost.


In [ ]:
print(f"Percentage of lost events: {((json_events_not_parsed_count - bot_count.value) / json_events_bak_count) * 100}")

Those events are lost as the Dataflow pipeline is unable to parse/process them

---

## Loading data stored in BigQuery

We need to fetch all the project ids to be able to read an `union` all the data from the different workspaces


In [ ]:
project_ids = json_events.select('projectId').distinct().rdd.map(lambda r: r[0]).collect()

In [ ]:
print(f"There are {len(project_ids)} in the `event_data`")

In [ ]:
def load_events_from_bigquery(project_id: str, event_date: str):
    return spark.read.format(
            'bigquery'
        ).load(
            f"""
            SELECT
              *
            FROM
              {project_id}.event
            WHERE
                DATE(eventDate) = '{event_date}'
            """
        )

def load_sessions_from_bigquery(project_id: str, session_start: str):
    return spark.read.format(
        'bigquery'
    ).load(
        f"""
            SELECT
              *
            FROM
              `{project_id}.session`
            WHERE
              DATE(sessionStart) = '{session_start}'
        """
    )

def load_projects(projects, _callable, _date):
    df = _callable(projects[0], _date)
    
    error_count = 0

    for project in projects[1:]:
        try:
            df = df.unionByName(_callable(project, _date), allowMissingColumns=True)
        except Exception as e:
            print(f'Error fetching: {project} - {str(e)}')
            error_count += 1

    print(f"Feched {len(projects) - error_count}, {error_count} errored when fetching")    

    return df.cache()

In [ ]:
bq_events = load_projects(project_ids, load_events_from_bigquery, event_date)

In [ ]:
bq_event_count = bq_events.count()
bq_event_count

 The number 1752733 is the event count achieved when processing data without windowing

In [ ]:
print(f"Missed events: {json_event_count - bq_event_count}")

However comparing the difference of events ingested by BiqQuery vs the number of events parsed by the Dataflow backfill pipeline is much smaller

In [ ]:
print(f"Percentage difference between BigQuery events vs Dataflow parsed events {((json_event_count - bq_event_count) / bq_event_count) * 100}")

---

## Comparing event sources

We can briefly inspect how the events compare between the different data sources:
* BigQuery

In [ ]:
bq_events.groupBy(
    'projectId', 'dataSourceId'
).agg(
    F.min('eventDate').alias('min_date'), 
    F.max('eventDate').alias('max_date'), 
    F.count('*').alias('count')
).orderBy(
    'projectId', 'dataSourceId'
).show(60, truncate=False)

* JSON Events parsed from backup data

In [ ]:
json_events.groupBy(
    'projectId', 'dataSourceId'
).agg(
    F.min('eventDate').alias('min_date'), 
    F.max('eventDate').alias('max_date'), 
    F.count('*').alias('count')
).orderBy(
    'projectId', 'dataSourceId'
).show(60, truncate=False)

Compare the impact on each workspace

In [ ]:
json_events.groupBy(
    'projectId', 'dataSourceId'
).agg(
    F.count('*').alias('json_count')
).join(
    bq_events.groupBy(
        'projectId', 'dataSourceId'
    ).agg(
        F.count('*').alias('bq_count')
    ),
    how='left',
    on=['projectId', 'dataSourceId']
).withColumn(
    'delta',
    F.expr('ABS(json_count - bq_count)')
).withColumn(
    'percent',
    F.expr('(delta/bq_count) * 100')
).orderBy(
    'percent', ascending=False
).show(60, truncate=False)

---
Inspect specific workspace

In [ ]:
workspace_id = 'asah8af32395b4f74ea59245d5b672ac6a84'

In [ ]:
json_events.select('sessionId', 'eventId', 'id', 'eventDate').filter(f'projectId = "{workspace_id}"').orderBy('eventDate').show(truncate=False)

In [ ]:
bq_events.select('sessionId', 'eventId', 'id', 'eventDate').filter(f'projectId = "{workspace_id}"').orderBy('eventDate').show(truncate=False)

In [ ]:
event_session_comparison = json_events.select(
    'projectId', 'id', 'eventDate', 'eventId', 'sessionId'
).join(
    bq_events.selectExpr(
        'projectId', 'id', 'eventDate', 'eventId', 'sessionId AS bqSessionId'
    ),
    on=['projectId', 'id', 'eventDate', 'eventId'],
    how='left'
).select(
    'id', 'eventDate', 'eventId', 'sessionId', 'bqSessionId'
).withColumn(
    'matched',
    F.when(
        F.col('sessionId') == F.col('bqSessionId'),
        True
    ).otherwise(
        False
    )
).orderBy(
    'eventDate'
).cache()

event_session_comparison.show(5, truncate=40)

The `event_session_comparison` dataframe allows us to learn how many session are not matched. This includes:
* Session ids mismatching
* Events who failed to process in the streaming pipeline 

In [ ]:
matched_sessions_count = event_session_comparison.filter('matched == true').count()
not_matched_sessions_count = event_session_comparison.filter('matched == false AND bqSessionId is not NULL').count()
missing_sessions_count = event_session_comparison.filter('matched == false AND bqSessionId is NULL').count()

print(f"""
    The process matches {matched_sessions_count}, however there are {not_matched_sessions_count} sessionIds mismatch and 
    {missing_sessions_count} events that do not have a session
    """
)

**Note**: It is clear there are differences in the Dataflow processing pipline leading to different session id attribution.

---

## Inspecting `sessions` BigQuery table vs Dataflow parsed one

However batch sessionisation did not work as expected. we need to work it out differently by loading existing session data and check if we can infer it from there.

In [ ]:
bq_sessions = load_projects(project_ids, load_sessions_from_bigquery, event_date)

We are creating a `json_events_new_session` dataframe by applying previously generated sessions as found in the `bq_events` dataframe

In [ ]:
bq_sessions_count = bq_sessions.count()
bq_sessions_count

In [ ]:
path = f"gs://{google_project}-dataflow/batch/output/{event_date.replace('-', '')}/sessions/*.jsonl"  # can include filenames with with ':'
paths_file = "events_json_path_list.txt"

!gsutil ls -r $path > $paths_file

path_list = open(paths_file, 'r').read().split("\n")[:-1]

json_sessions = spark.read.json(path_list)
json_sessions.cache()

json_session_count = json_sessions.count()
json_session_count

The number of sessions is different

In [ ]:
bq_sessions.filter(f'projectId = "{workspace_id}"').select('id', 'sessionStart', 'sessionEnd').orderBy('sessionStart').show(truncate=False)

In [ ]:
json_sessions.filter(f'projectId = "{workspace_id}"').select('id', 'sessionStart', 'sessionEnd').orderBy('sessionStart').show(truncate=False)

In [ ]:
bq_events.filter('sessionId = "7d84fa705fd00e9924ab7f2b6e0e163861b75bf9f880aafa45a84022e26ea41b"').count()

---

Attaching BigQuery `sessionId` to JSON parsed events

In [ ]:
json_events_new_session = json_events.drop(
    'sessionId'
).join(
    bq_events.select('id', 'projectId', 'dataSourceId', 'sessionId'),
    on=['id', 'projectId', 'dataSourceId'],
    how='left'
).cache() 

json_events_new_session.count()

In [ ]:
json_events_new_session.drop_duplicates().count()

In [ ]:
json_events_new_session.dropDuplicates(
    [c for c in json_events.columns if not c in ['createDate']]
).count()

We can count the number of events that are left without a sessons. This should match the event count of events parsed by Dataflow

In [ ]:
json_events_no_session = json_events_new_session.filter(
    'sessionId is NULL'
).cache() 

json_events_no_session.count()

In [ ]:
json_events_matched_sessions = json_events_no_session.join(
    bq_sessions.selectExpr('projectId', 'id AS sessionId_2', 'userId', 'sessionStart', 'sessionEnd'),
    on=['projectId', 'userId'],
    how='left'
).withColumn(
    'sessionMatch',
    F.when(
        F.col('eventDate').between(F.col('sessionStart'), F.col('sessionEnd')),
        True
    ).otherwise(
        False
    )
).filter('sessionMatch is true')

json_events_matched_sessions.count()

In [ ]:
json_events_no_matched_sessions = json_events_no_session.join(
    bq_sessions.selectExpr('projectId', 'id AS sessionId_2', 'userId', 'sessionStart', 'sessionEnd'),
    on=['projectId', 'userId'],
    how='left'
).withColumn(
    'sessionMatch',
    F.when(
        F.col('eventDate').between(F.col('sessionStart'), F.col('sessionEnd')),
        True
    ).otherwise(
        False
    )
).filter('sessionMatch is false')

json_events_no_matched_sessions.count()

In [ ]:
json_events_no_matched_sessions.select('id', 'eventDate', 'sessionStart', 'sessionEnd').orderBy('id', 'eventDate', 'sessionStart').show(truncate=False)

In [ ]:
json_events_new_session.join(
    json_events_matched_sessions.selectExpr('id', 'sessionId_2'),
    on=['id'],
    how='left'
).withColumn(
    'sessionId',
    F.coalesce(F.col('sessionId'), F.col('sessionId_2'))
).drop(
    'sessionId_2'
).count()#.filter('sessionId is  NULL').count()

---

In [ ]:
json_events_matched = json_events_bak.join(
    json_events.selectExpr('id', 'true AS matched'),
    on=['id'],
    how='left'
).cache()

In [ ]:
json_events_matched.groupBy('matched').count().show()

In [ ]:
json_events_dropped = json_events_matched.filter('matched is null')

In [ ]:
json_events_dropped.count()

In [ ]:
json_events_dropped.show(1)

In [ ]:
json_events_dropped.groupBy('projectId').count().orderBy('count', 'projectId',ascending=False).show(30, truncate=False)